In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import subprocess
from pathlib import Path

import anndata as ad
import numpy as np
import squidpy as sq
from mudata import MuData
from spatialleiden import spatialleiden_multiomics

from batch_integration.multisample_utils import (
    preprocess_neighbors,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, SHELL, activate_env_cmd, remove_tick_and_label

In [ ]:
spatialglue_dir = DATA_DIR / "SpatialGlue" / "Data_SpatialGlue"
spatialglue_results = RESULT_DIR / "SpatialGlue"

In [ ]:
spatialglue_results.mkdir(exist_ok=True, parents=True)

In [ ]:
seed = 42

# Prepare & Cluster

In [ ]:
for simulation in spatialglue_dir.glob("*Simulation*"):
    dataset = simulation.name.split("_")[1]
    modalities = {}

    # load modalities
    for file in simulation.glob("*.h5ad"):
        modality = file.stem.split("_")[1]
        adata = ad.read_h5ad(file)
        adata.X = adata.layers["counts"]
        del adata.layers, adata.uns, adata.varm
        modalities[modality] = adata

    # prepare MuData
    mdata = MuData(modalities)
    mdata.var_names_make_unique()

    mdata.obs["groundtruth"] = mdata.mod["RNA"].obsm["spfac"].astype(int) @ np.array(
        [1, 2, 3, 4]
    )
    mdata.obs["groundtruth"] = mdata.obs["groundtruth"].astype("category")

    mdata.obsm["spatial"] = mdata["RNA"].obsm["spatial"]

    for _, mod in mdata.mod.items():
        del mod.obsm["spatial"], mod.obsm["spfac"]

    # process modalities
    for name, adata in mdata.mod.items():
        n_pcs = 30 if name == "RNA" else 10
        # preprocess_genes(adata, batch_key=None, min_cells=1)
        preprocess_scaling(adata, n_pcs=n_pcs, seed=seed)
        preprocess_neighbors(adata, "X_pca", seed=seed)

    # spatial neighbors
    sq.gr.spatial_neighbors(mdata, coord_type="grid", n_neighs=8)

    # cluster
    spatialleiden_multiomics(
        mdata,
        resolution=0.6,
        layer_weights={"ADT": 1, "RNA": 1.5, "spatial": 2},
        seed=seed,
    )

    # write output
    mdata.write_h5mu(spatialglue_results / f"{dataset}.h5mu")

    labels = mdata.obs[["groundtruth", "spatialleiden"]]
    labels[["x", "y"]] = mdata.obsm["spatial"]
    labels.to_parquet(spatialglue_results / f"{dataset}.parquet")

# Run SpatialGlue

In [ ]:
conda_cmd = activate_env_cmd("spatialglue")

In [ ]:
script_dir = Path("other_methods")
script_path = script_dir / "spatialglue.py"

In [ ]:
for h5mu in spatialglue_results.glob("*.h5mu"):
    name = h5mu.stem
    out_dir = spatialglue_results / "SpatialGlue"
    out_dir.mkdir(exist_ok=True, parents=True)
    out_file = out_dir / f"{name}.parquet"

    cmd = f"{script_path} {h5mu} {out_file}"

    subprocess.run(f"{conda_cmd} && {cmd}", shell=True, executable=SHELL)

# Plotting & metrics

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

In [ ]:
scatter_kwargs = dict(
    # marker="s",
    size=1,
    lw=0,
    legend=False,
)

In [ ]:
methods = ["groundtruth", "spatialleiden", "SpatialGlue"]

In [ ]:
scores = []

for file in spatialglue_results.glob("*.parquet"):
    labels = (
        pd.read_parquet(file)
        .astype({"spatialleiden": "category", "groundtruth": "category"})
        .join(pd.read_parquet(file.parent / "SpatialGlue" / file.name))
        .astype({"SpatialGlue": "category"})
    )

    fig, axs = plt.subplots(
        ncols=len(methods), sharex=True, sharey=True, figsize=(9, 3)
    )
    for label, ax in zip(methods, axs):
        _ = sns.scatterplot(
            data=labels, x="x", y="y", hue=label, ax=ax, **scatter_kwargs
        )
        ax.set(title=label, aspect=1)
        remove_tick_and_label(ax)
    axs[0].set(ylabel=file.stem)
    fig.tight_layout()

    for tool in ["spatialleiden", "SpatialGlue"]:
        ari = adjusted_rand_score(
            labels["groundtruth"].cat.codes, labels[tool].cat.codes
        )
        nmi = normalized_mutual_info_score(
            labels["groundtruth"].cat.codes, labels[tool].cat.codes
        )

        scores.append({"sample": file.stem, "ARI": ari, "NMI": nmi, "tool": tool})

scores = pd.DataFrame(scores).set_index("sample")

In [ ]:
scores_df = scores.melt(
    id_vars=["tool"], var_name="metric", value_name="value", ignore_index=False
)

g = sns.catplot(data=scores_df, x="metric", y="value", hue="tool", kind="box")
_ = g.set(ylim=(0.5, 1))